In [ ]:
import pandas as pd
import numpy as np

# ============================================================================
# PASSO 1: Leitura base
# ============================================================================
df = pd.read_csv('Planeamento_15_07.csv', sep=';', encoding='utf-8-sig')

# Filtrar apenas entregas (DT)
df = df[df['Tipo'] == 'DT'].copy().reset_index(drop=True)

# Mapeamento direto
df['CP'] = df['Codigo Postal Destino'].str.strip()
df['GIRO'] = df['Ref.Entrega'].str[:5]
df['MORADA'] = df['Lugar de llegada'].fillna('')
df['LOCALIDADE'] = df['Ciudad Dest'].fillna('')
df['DATA_EVENTO'] = df['Fecha/hora real de descarga en destino'].str.split().str[-1]
df['DATA_CRIACAO'] = '15/07/2026'
df['PALETES'] = pd.to_numeric(df['Equivalent palettes Real (un)'], errors='coerce').fillna(0).astype(int)

# ============================================================================
# PASSO 2: Merge com Coordenadas (Clientes_SL.xlsx)
# ============================================================================
df_coords = pd.read_excel('Clientes_SL.xlsx', sheet_name='ACT_TMS')
df_coords_unique = df_coords[['CPOSTAL', 'DSTLAT', 'DSTLONG', 'LUGAR', 'DIRECCION', 'CIUDAD']].drop_duplicates(subset=['CPOSTAL'])

df = df.merge(
    df_coords_unique,
    left_on='CP',
    right_on='CPOSTAL',
    how='left'
)

df['LATITUDE'] = df['DSTLAT']
df['LONGITUDE'] = df['DSTLONG']
df['PLATAFORMA'] = df['LUGAR']

# Se não tiver morada no CSV, usar do Excel
df['MORADA'] = df['MORADA'].fillna(df['DIRECCION'])
df['LOCALIDADE'] = df['LOCALIDADE'].fillna(df['CIUDAD'])

# ============================================================================
# PASSO 3: Merge com Dados de Viaturas (SAL_DAT027)
# ============================================================================
df_viaturas = pd.read_excel('SAL_DAT027__16_.xls', sheet_name='SAL_DAT027', engine='xlrd')
df_viaturas['CODEUT'] = df_viaturas['CODEUT'].astype(str).str.strip()

df['CODEUT'] = df['GIRO'].astype(str)
df = df.merge(
    df_viaturas[['CODEUT', 'TRANSPORTISTA', 'TRACTORA', 'REMOLQUE', 'INGRESODT', 'COSTEDT', 'PALETSDT']],
    on='CODEUT',
    how='left'
)

# ============================================================================
# PASSO 4: Merge com Janelas Horárias (Book_2.xlsx)
# ============================================================================
df_janelas = pd.read_excel('Book_2.xlsx', sheet_name='Sheet1')

# Converter horas decimais para HH:MM
def decimal_to_time(decimal_hour):
    if pd.isna(decimal_hour):
        return '08:00'
    hours = int(decimal_hour * 24)
    return f"{hours:02d}:00"

df_janelas['Inicio'] = df_janelas['Janela Horaria inicio'].apply(decimal_to_time)
df_janelas['Fim'] = df_janelas['Janela Horaria inicio.1'].apply(decimal_to_time)

# Match por plataforma (LUGAR)
df = df.merge(
    df_janelas[['Plataforma', 'Inicio', 'Fim']],
    left_on='PLATAFORMA',
    right_on='Plataforma',
    how='left'
)

df['Inicio do Intervalo da Entrega'] = df['Inicio'].fillna('08:00')
df['Fim do Intervalo da Entrega'] = df['Fim'].fillna('17:00')

# ============================================================================
# PASSO 5: Capacidades e Paletes
# ============================================================================
# Capacidade 1 = Número de paletes (PALETES)
# Capacidade 2-10 = 1 (valor fixo)

for i in range(1, 11):
    if i == 1:
        df[f'Capacidade {i}'] = df['PALETES']  # Usar número real de paletes
    else:
        df[f'Capacidade {i}'] = 1  # Valor fixo

# ============================================================================
# PASSO 6: Adicionar campos fixos
# ============================================================================
df['CENTRO'] = 'Salvesen'
df['LOPTICA'] = 'DT'
df['COD_T_EVEN'] = ''
df['Grupo'] = 'DT'  # ou alguma classificação

# ============================================================================
# PASSO 7: Validações
# ============================================================================
# Remover linhas sem coordenadas
df = df[df['LATITUDE'].notna()]
df = df[df['LONGITUDE'] != 0]

# Remover duplicatas por nome/morada
df = df.drop_duplicates(subset=['MORADA', 'CP'], keep='first')

print(f"Total de entregas após processamento: {len(df)}")

# ============================================================================
# PASSO 8: Gerar XLSX
# ============================================================================
# Ler templates de Veículos, Regras, etc. do ficheiro original
df_veiculos = pd.read_excel('8818171.xlsx', sheet_name='Veículos')  # Do ficheiro config
df_regras = pd.read_excel('8818171.xlsx', sheet_name='Regras')
df_semireboque = pd.read_excel('8818171.xlsx', sheet_name='Semi-Reboques')

# Criar nova aba de veículos com dados de SAL_DAT027
df_veiculos_novo = df_viaturas[['CODEUT', 'TRANSPORTISTA', 'TRACTORA', 'REMOLQUE', 'INGRESODT', 'COSTEDT']].drop_duplicates()

# Gerar output
date_str = '20260715'
file_name = f"8818171_{date_str}_R_{len(df)}.xlsx"

with pd.ExcelWriter(file_name, engine='xlsxwriter') as writer:
    df_veiculos_novo.to_excel(writer, sheet_name='Veículos', index=False)
    df[['MORADA', 'CP', 'LOCALIDADE', 'LATITUDE', 'LONGITUDE', 'Inicio do Intervalo da Entrega', 
        'Fim do Intervalo da Entrega'] + [f'Capacidade {i}' for i in range(1, 11)] + 
        ['CENTRO', 'LOPTICA', 'Grupo']].to_excel(writer, sheet_name='Localizações', index=False)
    df_regras.to_excel(writer, sheet_name='Regras', index=False)
    df_semireboque.to_excel(writer, sheet_name='Semi-Reboques', index=False)

print(f"✅ XLSX gerado: {file_name}")